Imports & Setup

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

Paths

In [2]:
Q1_ROOT = "Q1"
Q2_TRAIN = "Q2/mnist_train.csv"
Q2_TEST = "Q2/mnist_test.csv"

IMG_SIZE = 128
BATCH_SIZE = 32


Q1 Cat & Dog Classification

Dataset & Transforms

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

train_dataset = datasets.ImageFolder(os.path.join(Q1_ROOT, "train"), transform=train_transform)
test_dataset  = datasets.ImageFolder(os.path.join(Q1_ROOT, "test"),  transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)
class_names


['Cat', 'Dog']

Simple CNN

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


ResNet18 Transfer Learning

In [5]:
def build_resnet18(num_classes):
    model = models.resnet18(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


Train & Evaluate Helpers

In [6]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    tot, correct, loss_sum = 0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * x.size(0)
        _, pred = out.max(1)
        correct += (pred == y).sum().item()
        tot += y.size(0)

    return loss_sum / tot, correct / tot


def evaluate(model, loader, criterion):
    model.eval()
    tot, correct, loss_sum = 0, 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            loss_sum += loss.item() * x.size(0)
            _, pred = out.max(1)
            correct += (pred == y).sum().item()
            tot += y.size(0)

    return loss_sum / tot, correct / tot


Train Simple CNN

In [7]:
simple_model = SimpleCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(simple_model.parameters(), lr=1e-3)

for epoch in range(5):
    tl, ta = train_one_epoch(simple_model, train_loader, criterion, optimizer)
    vl, va = evaluate(simple_model, test_loader, criterion)
    print(f"[SimpleCNN] Epoch {epoch+1} | Train Acc={ta:.3f} | Test Acc={va:.3f}")

torch.save(simple_model.state_dict(), "simple_cnn.pth")


[SimpleCNN] Epoch 1 | Train Acc=0.499 | Test Acc=0.500
[SimpleCNN] Epoch 2 | Train Acc=0.510 | Test Acc=0.500
[SimpleCNN] Epoch 3 | Train Acc=0.524 | Test Acc=0.500
[SimpleCNN] Epoch 4 | Train Acc=0.549 | Test Acc=0.600
[SimpleCNN] Epoch 5 | Train Acc=0.595 | Test Acc=0.600


Train ResNet18

In [8]:
resnet = build_resnet18(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.fc.parameters(), lr=1e-4)

for epoch in range(3):
    tl, ta = train_one_epoch(resnet, train_loader, criterion, optimizer)
    vl, va = evaluate(resnet, test_loader, criterion)
    print(f"[ResNet18] Epoch {epoch+1} | Train Acc={ta:.3f} | Test Acc={va:.3f}")

torch.save(resnet.state_dict(), "resnet18.pth")


c:\Users\param\Desktop\computervision\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\param\Desktop\computervision\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[ResNet18] Epoch 1 | Train Acc=0.601 | Test Acc=0.500
[ResNet18] Epoch 2 | Train Acc=0.724 | Test Acc=0.600
[ResNet18] Epoch 3 | Train Acc=0.793 | Test Acc=0.600


Inference Function

In [9]:
def predict_image(model, img_path):
    img = Image.open(img_path).convert("RGB")
    x = test_transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        out = model(x)
        _, pred = out.max(1)
    return class_names[pred.item()]


Test on Q1/Test + Internet Images

In [10]:
model = resnet  # choose best

# Test on provided test images
print("=== Q1/test predictions ===")
for cls in class_names:
    folder = os.path.join(Q1_ROOT, "test", cls)
    for f in os.listdir(folder):
        path = os.path.join(folder, f)
        if f.endswith(".jpg") or f.endswith(".png"):
            print(f, "TRUE:", cls, "| PRED:", predict_image(model, path))

# Internet images
internet_folder = "Q1/internet_test"
if os.path.exists(internet_folder):
    print("\n=== Internet images ===")
    for f in os.listdir(internet_folder):
        path = os.path.join(internet_folder, f)
        if f.lower().endswith(("jpg","png","jpeg")):
            print(f, "->", predict_image(model, path))
else:
    print("Create folder Q1/internet_test and add images.")


=== Q1/test predictions ===
Cat (1).jpg TRUE: Cat | PRED: Cat
Cat (2).jpg TRUE: Cat | PRED: Dog
Cat (3).jpg TRUE: Cat | PRED: Dog
Cat (4).jpg TRUE: Cat | PRED: Dog
Cat (5).jpg TRUE: Cat | PRED: Cat
Dog (1).jpg TRUE: Dog | PRED: Dog
Dog (2).jpg TRUE: Dog | PRED: Dog
Dog (3).jpg TRUE: Dog | PRED: Cat
Dog (4).jpg TRUE: Dog | PRED: Dog
Dog (5).jpg TRUE: Dog | PRED: Dog

=== Internet images ===
cat.jpg -> Cat
dog.jpg -> Dog


Q2 MNIST Classification

Load MNIST CSV

In [11]:
train_df = pd.read_csv(Q2_TRAIN)
test_df  = pd.read_csv(Q2_TEST)

y_train = train_df.iloc[:,0].values
X_train = train_df.iloc[:,1:].values / 255.0

y_test  = test_df.iloc[:,0].values
X_test  = test_df.iloc[:,1:].values / 255.0

X_train.shape, X_test.shape


((59999, 784), (9999, 784))

Logistic Regression

In [12]:
scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train)
X_test_lr  = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, multi_class="multinomial", n_jobs=-1)
logreg.fit(X_train_lr, y_train)

pred = logreg.predict(X_test_lr)
acc_logreg = accuracy_score(y_test, pred)
acc_logreg


c:\Users\param\Desktop\computervision\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


0.9217921792179218

KNN

In [13]:
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

# use subset because KNN is heavy
idx = np.random.choice(len(X_train), 15000, replace=False)
knn.fit(X_train[idx], y_train[idx])

test_idx = np.random.choice(len(X_test), 3000, replace=False)
pred_knn = knn.predict(X_test[test_idx])
acc_knn = accuracy_score(y_test[test_idx], pred_knn)
acc_knn


0.9553333333333334

Random Forest

In [14]:
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1)
idx = np.random.choice(len(X_train), 15000, replace=False)
rf.fit(X_train[idx], y_train[idx])

test_idx = np.random.choice(len(X_test), 3000, replace=False)
pred_rf = rf.predict(X_test[test_idx])
acc_rf = accuracy_score(y_test[test_idx], pred_rf)
acc_rf


0.9593333333333334